In [87]:
import pickle
import pandas as pd

In [88]:
path = f'prepared_data/dataset_for_feature_engineering.pickle'

with open(path, 'rb') as f:
    df = pickle.load(f)

In [89]:
df.shape

(144900, 7)

In [90]:
df["date"] = pd.to_datetime(df["date"])

In [91]:
df = df.sort_values(["ticker","date"]).reset_index(drop=True)

In [92]:
df.describe()

,date,open,high,low,close,volume
count,144900,144900.000000,144900.000000,144900.000000,144900.000000,1.449000e+05
mean,2020-09-24 05:06:12.968944384,567.670997,575.520949,558.834654,567.289519,1.952855e+08
min,2013-03-25 00:00:00,0.003915,0.003965,0.003800,0.003935,1.000000e+00
25%,2017-09-20 00:00:00,46.000000,46.812500,45.107500,45.980000,2.906800e+05
50%,2021-02-10 00:00:00,150.400000,152.865000,148.120000,150.370000,2.128706e+06
75%,2023-12-07 00:00:00,567.600000,576.600000,558.200000,567.000000,1.490144e+07
max,2026-03-06 00:00:00,9000.000000,10097.000000,8500.000000,8908.000000,6.616146e+11
std,NaN,1116.490278,1130.541507,1100.339082,1115.854818,2.954327e+09


In [93]:
# Доходность — это процентное изменение цены акции за выбранный период времени.
# Она показывает, насколько выросла или упала цена по сравнению с прошлым значением.

# Экономический смысл:
# Если доходность положительная — акция растёт.
# Если отрицательная — акция падает.
# Чем больше модуль доходности, тем сильнее движение цены.

df["ret_1"]  = df.groupby("ticker")["close"].pct_change(1)   # дневное изменение цены
df["ret_3"]  = df.groupby("ticker")["close"].pct_change(1)   # изменение цены за три дня
df["ret_5"]  = df.groupby("ticker")["close"].pct_change(5)   # изменение за торговую неделю
df["ret_20"] = df.groupby("ticker")["close"].pct_change(20)  # изменение за торговый месяц

In [94]:
# Волатильность показывает, насколько нестабильно ведёт себя цена акции.
# Чем выше волатильность — тем выше риск и тем более резкие движения цены происходят.
#
# Экономический смысл:
# Высокая волатильность означает нервный, неустойчивый рынок.
# Низкая — спокойное, стабильное движение.

df["vol3"]  = df.groupby("ticker")["ret_1"].rolling(3).std().reset_index(0, drop=True)   # волатильность за 3 дня
df["vol_5"]  = df.groupby("ticker")["ret_1"].rolling(5).std().reset_index(0, drop=True)   # волатильность за 5 дней
df["vol_15"] = df.groupby("ticker")["ret_1"].rolling(15).std().reset_index(0, drop=True) # волатильность за 15 дней
df["vol_30"] = df.groupby("ticker")["ret_1"].rolling(30).std().reset_index(0, drop=True) # волатильность за 30 дней
df["vol_45"] = df.groupby("ticker")["ret_1"].rolling(45).std().reset_index(0, drop=True) # волатильность за 45 дней
df["vol_90"] = df.groupby("ticker")["ret_1"].rolling(90).std().reset_index(0, drop=True) # волатильность за 90 дней

In [95]:
# Моментум показывает, насколько сильно и в каком направлении двигалась цена акции
# за выбранный период времени.
#
# Экономический смысл:
# Положительный моментум означает, что акция находится в фазе роста.
# Отрицательный — что акция находится в фазе падения.

df["mom_3"]  = df.groupby("ticker")["close"].pct_change(3)   # сила движения за 3 дня
df["mom_5"]  = df.groupby("ticker")["close"].pct_change(5)   # сила движения за 5 дней
df["mom_15"]  = df.groupby("ticker")["close"].pct_change(15)   # сила движения за 5 дней
df["mom_20"] = df.groupby("ticker")["close"].pct_change(20)  # сила движения за 20 дней

In [96]:
# Скользящие средние показывают сглаженное направление движения цены акции.
# Они позволяют отделить устойчивый тренд от случайных рыночных колебаний.
#
# Экономический смысл:
# Если короткая средняя выше длинной — тренд восходящий.
# Если ниже — нисходящий.

df["sma_3"]  = df.groupby("ticker")["close"].transform(lambda x: x.rolling(3).mean())    # средняя цена за 3 дня
df["sma_5"]  = df.groupby("ticker")["close"].transform(lambda x: x.rolling(5).mean())    # средняя цена за 5 дней
df["sma_10"]  = df.groupby("ticker")["close"].transform(lambda x: x.rolling(10).mean())    # средняя цена за 5 дней
df["sma_20"] = df.groupby("ticker")["close"].transform(lambda x: x.rolling(20).mean())   # средняя цена за 20 дней

df["sma_ratio"] = df["sma_5"] / df["sma_20"] - 1   # относительное положение краткой и длинной средних


In [97]:
# Объём торгов показывает активность участников рынка.
# Он отражает, насколько активно инвесторы покупают и продают акцию.
#
# Экономический смысл:
# Рост объёма подтверждает силу движения цены.
# Падение объёма — признак ослабления интереса к акции.

df["vol_mean_5"] = df.groupby("ticker")["volume"].transform(lambda x: x.rolling(5).mean())  # средний объём за 5 дней
df["vol_mean_15"] = df.groupby("ticker")["volume"].transform(lambda x: x.rolling(15).mean())  # средний объём за 15 дней
df["vol_mean_45"] = df.groupby("ticker")["volume"].transform(lambda x: x.rolling(45).mean())  # средний объём за 45 дней
df["vol_ratio"]  = df["volume"] / df["vol_mean_5"]                                          # относительный объём


In [98]:
# Свечной диапазон показывает внутридневную амплитуду колебаний цены.
# Он отражает уровень рыночной неопределённости и борьбу покупателей и продавцов.

df["hl_range"] = (df["high"] - df["low"]) / df["close"]   # относительный дневной диапазон


In [99]:
# Временные признаки отражают календарную структуру торговых сессий.
# Рынок ведёт себя по-разному в разные дни недели и месяцы года из-за поведения инвесторов,
# налоговых периодов, отчётностей и ребалансировок фондов.

# Базовые
df["dow"] = df["date"].dt.weekday                 # день недели (0 = пн, 6 = вс)
df["month"] = df["date"].dt.month                 # номер месяца (1-12)
df["week"] = df["date"].dt.isocalendar().week.astype(int)  # неделя года (1-52)
df["is_month_end"] = df["date"].dt.is_month_end.astype(int)
df["is_month_start"] = df["date"].dt.is_month_start.astype(int)

# Дополнительные признаки
df["day_of_month"] = df["date"].dt.day            # число месяца (1-31)
df["day_of_year"] = df["date"].dt.dayofyear       # день года (1-366)
df["quarter"] = df["date"].dt.quarter             # квартал (1-4)
df["is_quarter_end"] = df["date"].dt.is_quarter_end.astype(int)
df["is_quarter_start"] = df["date"].dt.is_quarter_start.astype(int)
df["is_weekend"] = (df["dow"] >= 5).astype(int)   # суббота/воскресенье = 1

# Для учёта цикличности (полезно для CatBoost, но можно и так)
df["dow_sin"] = np.sin(2 * np.pi * df["dow"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["dow"] / 7)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

In [100]:
# Целевая переменная показывает, вырастет ли цена акции на следующий торговый день.
# 1 — если завтрашняя цена закрытия выше сегодняшней.
# 0 — если цена не выросла.

df["target"] = (df.groupby("ticker")["close"].shift(-1) > df["close"]).astype(int)

In [101]:
info = ['date', 'ticker']

In [102]:
df.drop(columns=info).corr()['target'].sort_values()

dow_sin            -0.034717
ret_3              -0.019799
ret_1              -0.019799
mom_3              -0.018502
vol_90             -0.012086
vol_45             -0.009270
vol_ratio          -0.008701
vol_30             -0.007615
mom_5              -0.007543
ret_5              -0.007543
is_month_start     -0.007128
month              -0.006844
vol3               -0.006512
day_of_year        -0.006139
dow_cos            -0.006116
vol_mean_5         -0.006087
volume             -0.005762
vol_15             -0.005694
quarter            -0.005520
vol_5              -0.005220
week               -0.005211
mom_15             -0.003740
vol_mean_45        -0.003241
is_quarter_start   -0.002954
vol_mean_15        -0.002915
hl_range           -0.000759
ret_20              0.001077
mom_20              0.001077
close               0.001399
low                 0.001530
sma_3               0.001546
high                0.001556
open                0.001625
sma_5               0.001654
sma_10        

In [103]:
features = df.drop(columns=info+['target']).columns

In [104]:
df[df['ret_1']>1][['ticker', 'date', "close",'ret_1']]

,ticker,date,close,ret_1
135,AFKS,2014-12-18,13.630,1.062027
83659,PHOR,2013-08-15,1001.414,1.198750


In [105]:
def smooth_outliers(df, features, lower_percentile=0.1, upper_percentile=0.9):
    """
    Функция для сглаживания выбросов в заданных фичах.
    Заменяет значения, которые выходят за пределы интервала между нижним и верхним персентилем.
    
    :param df: pandas DataFrame, исходный датасет.
    :param features: список названий колонок (фич), в которых нужно проверить выбросы.
    :param lower_percentile: нижний персентиль, по умолчанию 0.1 (10-й персентиль).
    :param upper_percentile: верхний персентиль, по умолчанию 0.9 (90-й персентиль).
    :return: pandas DataFrame с заменёнными выбросами.
    """
    
    data = df.copy()
    
    for feature in features:
        if feature in data.columns:
            # Вычисляем персентиль для нижней и верхней границы
            lower_value = data[feature].quantile(lower_percentile)
            upper_value = data[feature].quantile(upper_percentile)
            
            # Заменяем значения, выходящие за пределы интервала
            data[feature] = data[feature].apply(lambda x: min(max(x, lower_value), upper_value))
    
    return data

In [106]:
df = smooth_outliers(df, features, lower_percentile=0.01, upper_percentile=0.99)

In [107]:
df[df['ret_1']>1][['ticker', 'date', "close",'ret_1']]

,ticker,date,close,ret_1


In [108]:
path = f'prepared_data/dataset_for_feature_selection.pickle'

samples = {
    'df'           : df,
    'info_fields'  : info,
    'features'     : features,
    'cat_features' : [],
    'num_features' : features,
    'target'       : 'target',
}
with open(path, 'wb') as f:
    pickle.dump(samples, f)

In [109]:
df.shape

(144900, 47)

In [110]:
df[features.tolist()+['target']].corr()['target'].sort_values()


dow_sin            -0.033565
ret_3              -0.019115
ret_1              -0.019115
mom_3              -0.015835
vol_90             -0.013475
vol_45             -0.010022
vol3               -0.007606
vol_ratio          -0.007462
vol_30             -0.007316
is_month_start     -0.007128
month              -0.006844
day_of_year        -0.006167
dow_cos            -0.006116
quarter            -0.005520
vol_5              -0.005267
week               -0.005236
volume             -0.003820
mom_5              -0.003715
ret_5              -0.003715
vol_15             -0.003288
vol_mean_5         -0.003234
vol_mean_45        -0.002920
vol_mean_15        -0.002263
hl_range           -0.001879
mom_15             -0.001784
close               0.001930
sma_3               0.002034
low                 0.002048
high                0.002077
open                0.002129
sma_5               0.002134
sma_20              0.002217
sma_10              0.002232
ret_20              0.003263
mom_20        